In [3]:
import torch
from model.actor_pretrainer import ActorPretrainer
from tokenizer.tokenizer import LabelTokenizer
from config.config import MODEL_CONFIG, PATH_CONFIG

C  = MODEL_CONFIG
PC = PATH_CONFIG

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# 加载 tokenizer 和模型结构
tokenizer  = LabelTokenizer(vocab_file=PC["vocab_file"])
vocab_size = tokenizer.get_vocab_size()
model      = ActorPretrainer(vocab_size=vocab_size).to(device)

# 加载权重
ckpt = torch.load(PC.get("ckpt_path", "best_model.pt"), map_location=device)
model.load_state_dict(ckpt["model"])
model.eval()  # ← 关键，关闭 dropout / batchnorm

print(f"✅ Loaded best model  loss={ckpt['loss']:.4f}  @ {ckpt['epoch_tag']}")


✅ Loaded best model  loss=4.2105  @ ep98


/tmp/ipykernel_7701/913752287.py:17: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  ckpt = torch.load(PC.get("ckpt_path", "best_model.pt"), map_location=device)


In [2]:
from torch_geometric.utils import to_dense_batch

@torch.no_grad()   # ← 关键，推理不需要梯度
def predict(batch):
    batch = batch.to(device)

    # 图编码
    node_embeddings, graph_embedding = model.graph_encoder(
        batch.x, batch.edge_index, batch.batch)
    graph_state   = model.state_proj(graph_embedding)
    decoder_state = model.state_tracker(batch.history_actions, graph_state)

    # 预测 action
    action_logits = model.action_predictor(decoder_state)
    pred_action   = action_logits.argmax(dim=-1)          # [B]

    # 预测 src / tgt
    dense_nodes, node_mask_bool = to_dense_batch(node_embeddings, batch.batch)
    src_logits, tgt_logits = model.pointer_network(
        decoder_state, dense_nodes, pred_action,
        node_mask=~node_mask_bool)
    pred_src = src_logits.argmax(dim=-1)                  # [B]
    pred_tgt = tgt_logits.argmax(dim=-1)                  # [B]

    # 预测 label（自回归解码）
    pred_label = greedy_decode_label(decoder_state, pred_action)

    return pred_action, pred_src, pred_tgt, pred_label

@torch.no_grad()
def greedy_decode_label(decoder_state, pred_action,
                        max_len=32):
    B      = decoder_state.size(0)
    # 用 BOS token 作为起始输入
    tokens = torch.full((B, 1), tokenizer.bos_token_id,
                        dtype=torch.long, device=device)

    for _ in range(max_len):
        logits     = model.label_decoder(decoder_state, pred_action, tokens)
        next_token = logits[:, -1, :].argmax(dim=-1, keepdim=True)  # [B, 1]
        tokens     = torch.cat([tokens, next_token], dim=-1)

        # 所有样本都生成了 EOS 则提前停止
        if (next_token.squeeze(-1) == tokenizer.eos_token_id).all():
            break

    return tokens[:, 1:]   # 去掉 BOS


In [3]:
from torch_geometric.loader import DataLoader
from data.ssr_graph_pretrain_dataset import SSRGraphDataset

dataset    = SSRGraphDataset(
    json_path=PC["test_data"], tokenizer=tokenizer,
    max_seq_len=C["max_seq_len"], max_hist_len=C["max_hist_len"],
)
dataloader = DataLoader(dataset, batch_size=32, shuffle=False)

correct_action = correct_src = correct_tgt = total = 0

for batch in dataloader:
    pred_action, pred_src, pred_tgt, _ = predict(batch)

    target_action = batch.target_action.squeeze(-1).to(device)
    target_src    = batch.target_src.squeeze(-1).to(device)
    target_tgt    = batch.target_tgt.squeeze(-1).to(device)

    correct_action += (pred_action == target_action).sum().item()
    correct_src    += (pred_src    == target_src).sum().item()
    correct_tgt    += (pred_tgt    == target_tgt).sum().item()
    total          += target_action.size(0)

print(f"Action Acc : {correct_action / total:.4f}")
print(f"Src    Acc : {correct_src    / total:.4f}")
print(f"Tgt    Acc : {correct_tgt    / total:.4f}")


Action Acc : 0.8748
Src    Acc : 0.2286
Tgt    Acc : 0.0892


In [5]:
import torch 
from torch_geometric.utils import to_dense_batch

@torch.no_grad()
def predict_oracle(batch):
    """用真实 action/src 作为输入，排除级联误差"""
    batch = batch.to(device)

    node_embeddings, graph_embedding = model.graph_encoder(
        batch.x, batch.edge_index, batch.batch)
    graph_state   = model.state_proj(graph_embedding)
    decoder_state = model.state_tracker(batch.history_actions, graph_state)

    target_action = batch.target_action.squeeze(-1)
    target_src    = batch.target_src.squeeze(-1)

    dense_nodes, node_mask_bool = to_dense_batch(node_embeddings, batch.batch)

    # 用真实 action 预测 src
    src_logits, tgt_logits = model.pointer_network(
        decoder_state, dense_nodes, target_action,
        target_src_idx=target_src,                  # ← 用真实 src 预测 tgt
        node_mask=~node_mask_bool)

    pred_src = src_logits.argmax(dim=-1)
    pred_tgt = tgt_logits.argmax(dim=-1)
    return pred_src, pred_tgt

from torch_geometric.loader import DataLoader
from data.ssr_graph_pretrain_dataset import SSRGraphDataset

dataset    = SSRGraphDataset(
    json_path=PC["test_data"], tokenizer=tokenizer,
    max_seq_len=C["max_seq_len"], max_hist_len=C["max_hist_len"],
)
dataloader = DataLoader(dataset, batch_size=32, shuffle=False)

correct_action = correct_src = correct_tgt = total = 0

for batch in dataloader:
    pred_src, pred_tgt = predict_oracle(batch)

    target_action = batch.target_action.squeeze(-1).to(device)
    target_src    = batch.target_src.squeeze(-1).to(device)
    target_tgt    = batch.target_tgt.squeeze(-1).to(device)

    correct_src    += (pred_src    == target_src).sum().item()
    correct_tgt    += (pred_tgt    == target_tgt).sum().item()
    total          += target_action.size(0)

print(f"Action Acc : {correct_action / total:.4f}")
print(f"Src    Acc : {correct_src    / total:.4f}")
print(f"Tgt    Acc : {correct_tgt    / total:.4f}")

Action Acc : 0.0000
Src    Acc : 0.2383
Tgt    Acc : 0.0969
